# Clinical Feature Analysis — QSVM Phase 1
**Task A — Prakshi**

Extract clinical features (CDR, MMSE, ASF, EDUC, SES), scale, reduce to 2 PCA components,
and save for multimodal QSVM training.

In [1]:
import pandas, numpy, matplotlib, seaborn, sklearn
print("all good")

all good


In [ ]:
# Cell 1 — Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')
print("✅ All imports successful")

In [ ]:
# Cell 2 — Shared Dataset Loading Block
# ============================================================
# SHARED DATASET LOADING BLOCK
# Copy verbatim into clinical_qsvm.ipynb, mri_qsvm.ipynb,
# and speech_qsvm.ipynb — do not change anything here
# ============================================================
import pandas as pd
import numpy as np

df = pd.read_csv('../data/multimodal_dementia_dataset.csv')

# Safety checks
assert df.shape == (225, 29), f"Unexpected shape: {df.shape}"
assert df.isnull().sum().sum() == 0, "NaNs found in dataset!"
assert df['Label'].isin([0, 1]).all(), "Invalid labels found!"

print(f"Dataset loaded: {df.shape}")
print(f"Label distribution: {df['Label'].value_counts()}")
print(f"Columns: {df.columns.tolist()}")
# ============================================================
# END SHARED LOADING BLOCK
# ============================================================

In [ ]:
# Cell 3 — Explore Clinical Features
CLINICAL_FEATURES = ['CDR', 'MMSE', 'ASF', 'EDUC', 'SES']

print("=== CLINICAL FEATURE OVERVIEW ===")
print(df[CLINICAL_FEATURES].describe().round(3))
print(f"Null values: {df[CLINICAL_FEATURES].isnull().sum().to_dict()}")

print("=== PER-CLASS MEANS ===")
print(df.groupby('Label')[CLINICAL_FEATURES].mean().round(3))

In [ ]:
# Cell 4 — Feature Distribution Plots
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
colors = {0: 'royalblue', 1: 'tomato'}
label_names = {0: 'Nondemented', 1: 'Demented'}

for ax, feat in zip(axes, CLINICAL_FEATURES):
    for lbl in [0, 1]:
        ax.hist(df[df['Label']==lbl][feat],
                bins=15, alpha=0.65, color=colors[lbl],
                label=label_names[lbl], edgecolor='black', linewidth=0.3)
    ax.set_title(feat, fontsize=11, fontweight='bold')
    ax.set_xlabel(feat)
    ax.set_ylabel('Count')
    ax.legend(fontsize=8)

plt.suptitle('Clinical Feature Distributions by Dementia Status',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('results/plots/clinical_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: results/plots/clinical_feature_distributions.png")

In [ ]:
# Cell 5 — Select Features + Scale
X_clinical = df[CLINICAL_FEATURES].values
y = df['Label'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_clinical)

print(f"Feature matrix shape: {X_scaled.shape}")
print(f"Mean after scaling (should be ~0): {X_scaled.mean(axis=0).round(4)}")
print(f"Std  after scaling (should be ~1): {X_scaled.std(axis=0).round(4)}")

In [ ]:
# Cell 6 — PCA Reduction to 2 Components
# Reduce 5 clinical features -> 2 principal components
# This is what gets passed to the QSVM (feature_dimension=2 per modality)
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print(f"Explained variance per component: {pca.explained_variance_ratio_.round(4)}")
print(f"Total variance retained: {pca.explained_variance_ratio_.sum():.2%}")
print(f"Shape after PCA: {X_pca.shape}")   # should be (225, 2)

# Visualize class separation in PCA space
plt.figure(figsize=(8, 5))
plt.scatter(X_pca[y==0, 0], X_pca[y==0, 1],
            c='royalblue', label='Nondemented', alpha=0.7, edgecolors='k', linewidths=0.3)
plt.scatter(X_pca[y==1, 0], X_pca[y==1, 1],
            c='tomato', label='Demented', alpha=0.7, edgecolors='k', linewidths=0.3)
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.title('PCA of Clinical Features (CDR, MMSE, ASF, EDUC, SES) Class Separation')
plt.legend()
plt.tight_layout()
plt.savefig('results/plots/clinical_pca.png', dpi=150)
plt.show()
print("Saved: results/plots/clinical_pca.png")

In [ ]:
# Cell 7 — Save PCA Features + Labels
# Save the 2D PCA-compressed clinical features
# Shape must be (225, 2) — Prakshi's multimodal notebook will concatenate this
np.save('results/X_clinical_pca.npy', X_pca)

# y_labels saved ONCE here (all modalities share the same label vector)
np.save('results/y_labels.npy', y)

print(f"✅ Saved: results/X_clinical_pca.npy  — shape: {X_pca.shape}")
print(f"✅ Saved: results/y_labels.npy         — shape: {y.shape}")

# Sanity check
check_X = np.load('results/X_clinical_pca.npy')
check_y = np.load('results/y_labels.npy')
assert check_X.shape == (225, 2), f"Wrong shape: {check_X.shape}"
assert check_y.shape == (225,),   f"Wrong shape: {check_y.shape}"
print("✅ Reload check passed")
print(f"   X_clinical_pca: {check_X.shape}  |  y_labels: {check_y.shape}")